# Apriori vs. FP-Growth on the COVID-19 Spread Chain Binary Matrix

This notebook applies **both Apriori and FP-Growth** to `Binary_Matrix_of_Spread_Chain.csv`
in one place, so their results can be compared directly. As before, the **only** preprocessing
step is removing the fully empty rows in the file — no other cleaning, filtering, or value
changes are made anywhere in this notebook.

**What the data represents:** each row is a *source country*, and each of the 186 destination-country
columns is `1` if the spread chain from that source reached that destination, `0` otherwise.
Each row is a "transaction" (a source country's spread event), and each destination country that
received the spread is an "item" in that transaction. Both algorithms are solving the same
problem — finding destination countries that frequently appear *together* in the same spread
chain — just via different mechanics, which is what the comparison at the end looks at.

In [1]:
# Step 0 — Install the library both algorithms come from
!pip install mlxtend --quiet

**What this does:** `mlxtend` (Machine Learning Extensions) provides standard, tested
implementations of both `apriori` and `fpgrowth`, plus a shared `association_rules` function
that works on the output of either one. Using the same library for both keeps the comparison
fair — any difference in the results comes from the algorithm, not from two different codebases.

In [2]:
# Step 1 — Upload the CSV to this Colab session
from google.colab import files
uploaded = files.upload()   # choose Binary_Matrix_of_Spread_Chain.csv when prompted

filename = list(uploaded.keys())[0]
print("Uploaded file:", filename)

Saving Binary Matrix of Spread Chain.csv to Binary Matrix of Spread Chain.csv
Uploaded file: Binary Matrix of Spread Chain.csv


**What this does:** Colab runs on a remote machine with no access to your local disk, so
the file must be uploaded into the session's temporary storage first. `files.upload()` opens a
file picker; once you select the CSV it's saved into the notebook's working directory.

In [3]:
# Step 2 — Load the raw CSV
import pandas as pd

df = pd.read_csv(filename)
print("Shape as loaded:", df.shape)
df.head()

Shape as loaded: (42, 187)


,Source Country,Afghanistan,Albania,Algeria,Andorra,Angola,Antigua and Barbuda,Argentina,Armenia,Australia,...,Uruguay,Uzbekistan,Vatican City,Venezuela,Vietnam,West Bank and Gaza,Western Sahara,Yemen,Zambia,Zimbabwe
0,Italy,0,1,1,1,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0
1,China,0,0,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
2,Iran,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,United Kingdom,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,France,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0


**What this does:** reads the file into a pandas DataFrame with no modification —
every row and column is loaded exactly as it appears in the CSV.

In [4]:
# Step 3 — Preprocessing: remove empty rows only
# The file has trailing rows with no Source Country and no data in any column at all.
# This is the only preprocessing step performed: drop rows with nothing in them.
df = df.dropna(subset=['Source Country'])

print("Shape after removing empty rows:", df.shape)
df.head()

Shape after removing empty rows: (42, 187)


,Source Country,Afghanistan,Albania,Algeria,Andorra,Angola,Antigua and Barbuda,Argentina,Armenia,Australia,...,Uruguay,Uzbekistan,Vatican City,Venezuela,Vietnam,West Bank and Gaza,Western Sahara,Yemen,Zambia,Zimbabwe
0,Italy,0,1,1,1,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0
1,China,0,0,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
2,Iran,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,United Kingdom,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,France,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0


**What this does:** drops rows where `Source Country` is empty. In this file, those rows
have no values in any column, so this removes rows with no data rather than changing, filtering,
or reinterpreting any actual `0`/`1` value. No other step is applied — every remaining cell keeps
its original value, and this same cleaned table is used for **both** algorithms below, so neither
one gets an advantage from different input data.

In [5]:
# Step 4 — Format for the frequent-itemset functions
# Both apriori() and fpgrowth() need 'Source Country' set aside as the transaction label
# (not treated as an item), and the item columns as True/False. Format conversion only.
transactions = df.set_index('Source Country').astype(bool)

print("Transactions shape (source countries x destination countries):", transactions.shape)
transactions.head()

Transactions shape (source countries x destination countries): (42, 186)


,Afghanistan,Albania,Algeria,Andorra,Angola,Antigua and Barbuda,Argentina,Armenia,Australia,Austria,...,Uruguay,Uzbekistan,Vatican City,Venezuela,Vietnam,West Bank and Gaza,Western Sahara,Yemen,Zambia,Zimbabwe
Source Country,,,,,,,,,,,,,,,,,,,,,
Italy,False,True,True,True,False,False,True,False,False,True,...,True,False,False,False,False,False,False,False,False,False
China,False,False,False,False,False,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
Iran,True,False,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
United Kingdom,False,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
France,False,False,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,True,False


**What this does:** `Source Country` becomes the row label (the transaction ID), and the
186 remaining columns (destination countries) become items. Each cell keeps the same 0/1
information from the file, just typed as `True`/`False`, which both `apriori()` and `fpgrowth()`
require internally. This single `transactions` table is what both algorithms run on.

In [6]:
# Step 5 — Run Apriori
import time
from mlxtend.frequent_patterns import apriori

start = time.time()
frequent_itemsets_apriori = apriori(transactions, min_support=0.04, use_colnames=True)
time_apriori = time.time() - start

frequent_itemsets_apriori = frequent_itemsets_apriori.sort_values(by='support', ascending=False)
print(f"Apriori found {len(frequent_itemsets_apriori)} frequent itemsets in {time_apriori:.4f} seconds")
frequent_itemsets_apriori.head(10)

Apriori found 25 frequent itemsets in 0.0073 seconds


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,support,itemsets
20,0.095238,(Togo)
16,0.071429,(Peru)
22,0.071429,(West Bank and Gaza)
15,0.071429,(Niger)
4,0.047619,(Cyprus)
1,0.047619,(Benin)
2,0.047619,(Botswana)
3,0.047619,(Burundi)
0,0.047619,(Bahrain)
8,0.047619,(Honduras)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**What this does:** Apriori repeatedly scans the transaction table, generating candidate
itemsets of increasing size and directly checking each one's support against the data. It uses
the "apriori property" for efficiency — if an itemset is infrequent, every larger itemset built
from it must also be infrequent, so those branches are pruned instead of being checked. The run
is timed so it can be compared against FP-Growth's runtime later.

- `min_support=0.04`: with only 42 transactions, an itemset appearing in just 2 rows already has
  support = 2/42 ≈ 0.048, so this threshold needs to stay low to find anything at all.

In [7]:
# Step 6 — Run FP-Growth
from mlxtend.frequent_patterns import fpgrowth

start = time.time()
frequent_itemsets_fpgrowth = fpgrowth(transactions, min_support=0.04, use_colnames=True)
time_fpgrowth = time.time() - start

frequent_itemsets_fpgrowth = frequent_itemsets_fpgrowth.sort_values(by='support', ascending=False)
print(f"FP-Growth found {len(frequent_itemsets_fpgrowth)} frequent itemsets in {time_fpgrowth:.4f} seconds")
frequent_itemsets_fpgrowth.head(10)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

FP-Growth found 25 frequent itemsets in 0.0081 seconds


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,support,itemsets
12,0.095238,(Togo)
20,0.071429,(Niger)
23,0.071429,(West Bank and Gaza)
13,0.071429,(Peru)
4,0.047619,(Saudi Arabia)
1,0.047619,(Saint Kitts and Nevis)
2,0.047619,(Portugal)
3,0.047619,(Cyprus)
0,0.047619,(Ukraine)
8,0.047619,(Zimbabwe)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**What this does:** FP-Growth scans the data twice, builds a compressed tree structure
(the "FP-tree") that encodes all the itemset/frequency information, and mines frequent itemsets
directly from that tree — without generating and testing candidate itemsets one at a time the
way Apriori does. It's run on the exact same `transactions` table and `min_support`, so the two
results are directly comparable.

In [8]:
# Step 7 — Generate association rules from each algorithm's itemsets
from mlxtend.frequent_patterns import association_rules

rules_apriori = association_rules(frequent_itemsets_apriori, metric="confidence", min_threshold=0.5)
rules_apriori = rules_apriori.sort_values(by='lift', ascending=False)

rules_fpgrowth = association_rules(frequent_itemsets_fpgrowth, metric="confidence", min_threshold=0.5)
rules_fpgrowth = rules_fpgrowth.sort_values(by='lift', ascending=False)

print(f"Apriori rules: {len(rules_apriori)}   FP-Growth rules: {len(rules_fpgrowth)}")
rules_apriori[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Apriori rules: 2   FP-Growth rules: 2


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,antecedents,consequents,support,confidence,lift
0,(Myanmar),(Kenya),0.047619,1.0,21.0
1,(Kenya),(Myanmar),0.047619,1.0,21.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**What this does:** turns each algorithm's frequent itemsets into if-then rules of the
form **{A} -> {B}** ("if the spread reached country A, it also reached country B"), each with:

- **support** — how often A and B occur together, across all source countries
- **confidence** — of the source countries where A occurred, what fraction also had B (P(B|A))
- **lift** — how much more likely B is *given* A, versus B occurring on its own
  (lift > 1 = positive association, lift = 1 = independent, lift < 1 = negative association)

The same `association_rules()` function and the same `min_threshold=0.5` are used for both, so
any difference in the rule tables below reflects the upstream itemsets, not the rule-generation
step.

In [9]:
# Step 8 — Compare the two algorithms
print("="*60)
print("COMPARISON: Apriori vs. FP-Growth")
print("="*60)

print(f"\nRuntime:")
print(f"  Apriori:    {time_apriori:.4f} seconds")
print(f"  FP-Growth:  {time_fpgrowth:.4f} seconds")

print(f"\nFrequent itemsets found:")
print(f"  Apriori:    {len(frequent_itemsets_apriori)}")
print(f"  FP-Growth:  {len(frequent_itemsets_fpgrowth)}")

print(f"\nAssociation rules generated:")
print(f"  Apriori:    {len(rules_apriori)}")
print(f"  FP-Growth:  {len(rules_fpgrowth)}")

# Check whether the two algorithms found the exact same itemsets (they should, by definition)
itemsets_apriori = set(frozenset(s) for s in frequent_itemsets_apriori['itemsets'])
itemsets_fpgrowth = set(frozenset(s) for s in frequent_itemsets_fpgrowth['itemsets'])
same_itemsets = itemsets_apriori == itemsets_fpgrowth

print(f"\nIdentical frequent itemsets found by both algorithms: {same_itemsets}")

COMPARISON: Apriori vs. FP-Growth

Runtime:
  Apriori:    0.0073 seconds
  FP-Growth:  0.0081 seconds

Frequent itemsets found:
  Apriori:    25
  FP-Growth:  25

Association rules generated:
  Apriori:    2
  FP-Growth:  2

Identical frequent itemsets found by both algorithms: True


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

**What this does, and what to expect:** Apriori and FP-Growth are two different *search
strategies* for the same well-defined problem, so — given the same data and the same
`min_support` — they are mathematically guaranteed to return the **same set of frequent
itemsets** every time. The `same_itemsets` check above should print `True`; if it doesn't,
that would point to a bug in the notebook rather than a real algorithmic difference, since
FP-Growth is not an approximation of Apriori's result.

Where they genuinely differ is **how** they get there:

| | Apriori | FP-Growth |
|---|---|---|
| Approach | Generates candidate itemsets level by level, checks each against the full data | Builds one compressed tree (FP-tree), mines itemsets directly from it |
| Data scans | Multiple (one per itemset size) | Two, total |
| Candidate generation | Yes — explicitly creates and tests candidates | No — itemsets are read off the tree structure |
| Typical performance | Slower as the number of items/transactions grows | Faster on larger or denser datasets, since it avoids repeated full scans |
| Result | Same frequent itemsets, same rules | Same frequent itemsets, same rules |

On this dataset (42 transactions, 186 items) the runtime difference above will be small — that's
expected, since FP-Growth's advantage shows up mainly on much larger datasets. The meaningful
result to report is that **both algorithms agree** on the frequent itemsets and rules, which is
a legitimate cross-validation of the pattern-mining step to mention in the paper's methods or
results section.

In [10]:
# Step 9 — Show the strongest rules side by side
print("Top Apriori rules:\n")
for _, row in rules_apriori.sort_values(['lift','confidence'], ascending=False).head(5).iterrows():
    a = ', '.join(list(row['antecedents'])); c = ', '.join(list(row['consequents']))
    print(f"  {a} -> {c}   (support={row['support']:.3f}, confidence={row['confidence']:.3f}, lift={row['lift']:.3f})")

print("\nTop FP-Growth rules:\n")
for _, row in rules_fpgrowth.sort_values(['lift','confidence'], ascending=False).head(5).iterrows():
    a = ', '.join(list(row['antecedents'])); c = ', '.join(list(row['consequents']))
    print(f"  {a} -> {c}   (support={row['support']:.3f}, confidence={row['confidence']:.3f}, lift={row['lift']:.3f})")

Top Apriori rules:

  Myanmar -> Kenya   (support=0.048, confidence=1.000, lift=21.000)
  Kenya -> Myanmar   (support=0.048, confidence=1.000, lift=21.000)

Top FP-Growth rules:

  Myanmar -> Kenya   (support=0.048, confidence=1.000, lift=21.000)
  Kenya -> Myanmar   (support=0.048, confidence=1.000, lift=21.000)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

**What this does:** prints the top 5 rules from each algorithm's output side by side in
plain-English form. With only 42 transactions, note that very high lift values (e.g. 20+)
reflect the small sample size as much as a strong underlying pattern — treat standout rules as
candidates worth investigating further rather than firm conclusions on their own.

In [11]:
# Step 10 (optional) — Save all results for use in the paper
frequent_itemsets_apriori.to_csv('apriori_frequent_itemsets.csv', index=False)
rules_apriori.to_csv('apriori_association_rules.csv', index=False)
frequent_itemsets_fpgrowth.to_csv('fpgrowth_frequent_itemsets.csv', index=False)
rules_fpgrowth.to_csv('fpgrowth_association_rules.csv', index=False)

from google.colab import files
files.download('apriori_frequent_itemsets.csv')
files.download('apriori_association_rules.csv')
files.download('fpgrowth_frequent_itemsets.csv')
files.download('fpgrowth_association_rules.csv')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

**What this does:** exports the frequent itemsets and rules from both algorithms to
separate CSV files and downloads them — useful for building a comparison table/figure in the
paper, or feeding either result into further analysis (e.g. the network-analysis / PrefixSpan
part of the project).